In [1]:
import pandas as pd
import yfinance as yf
from sqlalchemy import create_engine, text

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

trading


In [2]:
raw = yf.download("SPY", start="2005-01-01", end="2026-01-01",
                  auto_adjust=False, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw.head()

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,80.973511,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,79.984100,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,79.432152,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,79.835999,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,79.721565,118.440002,119.230003,118.129997,118.970001,55847700


In [3]:
df = (raw.reset_index()
         .rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low",
                          "Close": "close", "Adj Close": "adj_close", "Volume": "volume"}))
df["ticker"] = "SPY"
df = df[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]]
print(df.shape, df["date"].min(), df["date"].max())
print(df.isna().sum())

(5283, 8) 2005-01-03 00:00:00 2025-12-31 00:00:00
Price
ticker       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64


In [4]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS prices (
            ticker     TEXT    NOT NULL,
            date       DATE    NOT NULL,
            open       NUMERIC, high NUMERIC, low NUMERIC,
            close      NUMERIC, adj_close NUMERIC,
            volume     BIGINT,
            PRIMARY KEY (ticker, date)
        )
    """))

In [5]:
df.columns.name = None
df.to_sql("prices", engine, if_exists="append", index=False)

283

In [6]:
pd.read_sql("""
    SELECT ticker, COUNT(*) AS n_rows, MIN(date), MAX(date)
    FROM prices GROUP BY ticker
""", engine)

,ticker,n_rows,min,max
0,SPY,5283,2005-01-03,2025-12-31


In [7]:
def load_ticker(ticker):
    raw = yf.download(ticker, start="2005-01-01", end="2026-01-01",
                      auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    out = (raw.reset_index()
              .rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low",
                               "Close": "close", "Adj Close": "adj_close", "Volume": "volume"}))
    out["ticker"] = ticker
    out = out[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]]
    out.columns.name = None
    print(ticker, out.shape, out.isna().sum().sum(), "nulls")
    out.to_sql("prices", engine, if_exists="append", index=False)

for t in ["QQQ", "GLD"]:
    load_ticker(t)

QQQ (5283, 8) 0 nulls
GLD (5283, 8) 0 nulls
